In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

### Data Loading

In [2]:
df1 = pd.read_csv("dataset1.csv")
print(f"Dataset1 loaded: {len(df1)} rows, {len(df1.columns)} columns")

df2a = pd.read_csv("dataset2a for q1 q2.csv")
print(f"Dataset2a loaded: {len(df2a)} rows, {len(df2a.columns)} columns")

df2b = pd.read_csv("dataset2b for q3 q4.csv")
print(f"Dataset2b loaded: {len(df2b)} rows, {len(df2b.columns)} columns")

df3 = pd.read_csv("dataset3.csv")
print(f"Dataset3 loaded: {len(df3)} rows, {len(df3.columns)} columns")


Dataset1 loaded: 3000 rows, 17 columns
Dataset2a loaded: 546000 rows, 5 columns
Dataset2b loaded: 552000 rows, 5 columns
Dataset3 loaded: 6429 rows, 11 columns


### Dataset 1 Cleaning - SalesForce data

In [3]:
# 1.1 Handle missing values
missing_df1 = df1.isnull().sum()
missing_df1 = missing_df1[missing_df1 > 0].sort_values(ascending=False)
missing_df1

contract_end_date    1428
industry              862
dtype: int64

In [4]:
# contract_end_date is allowed to have null values; for industry, the severity of missing is quite significant, hence choose imputation
# Fill null values with 'Unknown' 
df1['industry'] = df1['industry'].fillna('Unknown')
print("Filled missing industry with 'Unknown'")

Filled missing industry with 'Unknown'


In [5]:
# 1.2 Check duplicate records
print("\n--- Duplicate check ---")
print(f"Unique customers: {df1['customer_id'].nunique()}")
print(f"Total records: {len(df1)}")
if df1['customer_id'].nunique() != len(df1):
    print("Warning: duplicate customer_id found")
    duplicates = df1[df1.duplicated(subset=['customer_id'], keep=False)]
    print(f"Duplicate records: {len(duplicates)}")
else:
    print("No duplicate customer_id")


--- Duplicate check ---
Unique customers: 3000
Total records: 3000
No duplicate customer_id


In [6]:
# 1.3 Handle date fields
print("\n--- Handling date fields ---")
df1['contract_start_date'] = pd.to_datetime(df1['contract_start_date'], format='%m/%d/%Y', errors='coerce')
df1['contract_end_date'] = pd.to_datetime(df1['contract_end_date'], format='%m/%d/%Y', errors='coerce')
print("Date fields converted to datetime object")


--- Handling date fields ---
Date fields converted to datetime object


In [7]:
# Save cleaned dataset1
df1_cleaned = df1.copy()
df1_cleaned.to_csv('dataset1_cleaned.csv', index=False)
print("Dataset1 cleaning completed, saved as 'dataset1_cleaned.csv'")

Dataset1 cleaning completed, saved as 'dataset1_cleaned.csv'


### Dataset 2 Cleaning - Usage logs

In [8]:
# 2.1 Merge the two log datasets
print("\n--- Merging logs ---")
df2a['date'] = pd.to_datetime(df2a['date'])
df2b['date'] = pd.to_datetime(df2b['date'])
print(f"Dataset2a date range: {df2a['date'].min()} to {df2a['date'].max()}")
print(f"Dataset2b date range: {df2b['date'].min()} to {df2b['date'].max()}")

df2_combined = pd.concat([df2a, df2b], ignore_index=True)
print(f"Combined records: {len(df2_combined)}")


--- Merging logs ---
Dataset2a date range: 2024-01-01 00:00:00 to 2024-06-30 00:00:00
Dataset2b date range: 2024-07-01 00:00:00 to 2024-12-31 00:00:00
Combined records: 1098000


In [9]:
# 2.2 Handle missing values
# Inspect missing values
missing_df2 = df2_combined.isnull().sum()
missing_df2 = missing_df2[missing_df2 > 0]
if len(missing_df2) > 0:
    for col, count in missing_df2.items():
        print(f"  {col}: {count}")
else:
    print("  No missing values")


  logins: 56010
  feature_events: 56010
  session_minutes: 56010


In [10]:
# All three needs to be addressed; insignificant in magnitude but NOT missing at random (EU customers' September logs) -> Imputation
# Time-series data -> Use interpolation
login_minus_31 = df2_combined['logins'].shift(31)
login_plus_30  = df2_combined['logins'].shift(-30)
mask = df2_combined['logins'].isna()
df2_combined.loc[mask, 'logins'] = (login_minus_31[mask] + login_plus_30[mask]) / 2

feature_events_minus_31 = df2_combined['feature_events'].shift(31)
feature_events_plus_30  = df2_combined['feature_events'].shift(-30)
mask = df2_combined['feature_events'].isna()
df2_combined.loc[mask, 'feature_events'] = (feature_events_minus_31[mask] + feature_events_plus_30[mask]) / 2

session_minutes_minus_31 = df2_combined['session_minutes'].shift(31)
session_minutes_plus_30  = df2_combined['session_minutes'].shift(-30)
mask = df2_combined['session_minutes'].isna()
df2_combined.loc[mask, 'session_minutes'] = (session_minutes_minus_31[mask] + session_minutes_plus_30[mask]) / 2

In [11]:
# 2.3 Check anomalies
print(f"\nAnomaly checks:")
print(f"  Negative logins: {(df2_combined['logins'] < 0).sum()}")
print(f"  Negative session minutes: {(df2_combined['session_minutes'] < 0).sum()}")


Anomaly checks:
  Negative logins: 0
  Negative session minutes: 0


In [13]:
# Save cleaned dataset2
df2_cleaned = df2_combined.copy()
df2_cleaned.to_csv('dataset2_cleaned.csv', index=False)
print("Dataset2 cleaned and saved to 'dataset2_cleaned.csv'")

Dataset2 cleaned and saved to 'dataset2_cleaned.csv'


### Dataset 3 Cleaning - Support Tickets

In [14]:
# 1.1 Data quality checks for tickets
print(f"Total tickets: {len(df3)}")
print(f"Unique customers: {df3['customer_id'].nunique()}")
print("Missing values summary:")
missing_df3 = df3.isnull().sum()
missing_df3 = missing_df3[missing_df3 > 0]
if len(missing_df3) > 0:
    for col, count in missing_df3.items():
        print(f"  {col}: {count}")
else:
    print("  No missing values")


Total tickets: 6429
Unique customers: 2421
Missing values summary:
  No missing values


In [15]:
# 1.2 Handle datetime fields
df3['created_at'] = pd.to_datetime(df3['created_at'], format='%Y-%m-%dT%H:%M', errors='coerce')
print("Datetime fields converted")
print(f"Ticket time range: {df3['created_at'].min()} to {df3['created_at'].max()}")

Datetime fields converted
Ticket time range: 2024-01-05 15:12:00 to 2024-12-30 23:28:00


In [16]:
# 1.3 Check issue category distribution
print("Issue category distribution:")
print(df3['issue_category'].value_counts())

Issue category distribution:
issue_category
billing_admin          1671
product_usability      1622
product_performance    1592
sales_expectation      1544
Name: count, dtype: int64


In [17]:
# Save cleaned dataset3
df3_cleaned = df3.copy()
df3_cleaned.to_csv('dataset3_cleaned.csv', index=False)
print("Dataset3 cleaned and saved to 'dataset3_cleaned.csv'")

Dataset3 cleaned and saved to 'dataset3_cleaned.csv'
